# Hindi/Punjabi WhatsApp Fake News Detection - Kaggle Training Notebook

This notebook trains the complete model bundle for WhatsApp-style Hindi/Punjabi fake-news detection. It expects the prepared dataset to be uploaded to Kaggle as an input dataset named `whatsapp-fake-news` and saves all artifacts to `/kaggle/working/whatsapp_model_artifacts.zip`.

Trained models:
- Logistic Regression + TF-IDF
- Naive Bayes + TF-IDF
- XGBoost + TF-IDF + WhatsApp metadata features
- IndicBERT: `ai4bharat/indic-bert`
- Punjabi BERT: tries `neuralspace-reverie/indic-transformers-pa-bert`, falls back to `l3cube-pune/punjabi-bert`
- MuRIL: `google/muril-base-cased`
- XGBoost stacking ensemble over available base-model probabilities + metadata

In [ ]:
# Kaggle usually includes most of these packages. Install only missing packages to avoid slow forced upgrades.
import importlib.util
import subprocess
import sys

PACKAGE_IMPORTS = {
    'transformers': 'transformers',
    'datasets': 'datasets',
    'evaluate': 'evaluate',
    'accelerate': 'accelerate',
    'scikit-learn': 'sklearn',
    'xgboost': 'xgboost',
    'joblib': 'joblib',
    'scipy': 'scipy',
    'sentencepiece': 'sentencepiece',
    'protobuf': 'google.protobuf',
}
missing_packages = [pkg for pkg, module in PACKAGE_IMPORTS.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages], check=False)
else:
    print('All required Python packages are already importable.')

In [ ]:
import gc
import inspect
import json
import math
import os
import random
import re
import shutil
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import evaluate
from datasets import Dataset
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

RUN_CLASSICAL = True
RUN_TRANSFORMERS = True
TRAIN_STACKING = True

MAX_LENGTH = 128
EPOCHS = 3
LEARNING_RATE = 2e-5
BATCH_SIZE = 16

SAVE_DIR = Path('/kaggle/working/whatsapp_model_artifacts')
CLASSICAL_DIR = SAVE_DIR / 'classical'
TRANSFORMER_DIR = SAVE_DIR / 'transformers'
ENSEMBLE_DIR = SAVE_DIR / 'ensemble'
for directory in [SAVE_DIR, CLASSICAL_DIR, TRANSFORMER_DIR, ENSEMBLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
REQUIRED_COLUMNS = ['text', 'label', 'language', 'source_type']

DATASET_CANDIDATE_DIRS = [
    Path('/kaggle/input/whatsapp-fake-news/whatsapp_dataset'),
    Path('/kaggle/input/whatsapp-fake-news'),
    Path('/kaggle/input/whatsapp-fake-news/whatsapp_fake_news/data/whatsapp_dataset'),
    Path('/kaggle/input/whatsapp-fake-news/data/whatsapp_dataset'),
    Path('/kaggle/working/whatsapp_dataset'),
]

def resolve_dataset_dir():
    for candidate in DATASET_CANDIDATE_DIRS:
        if all((candidate / f'whatsapp_{split}.csv').exists() for split in ['train', 'valid', 'test']):
            return candidate
    searched = '\n'.join(str(path) for path in DATASET_CANDIDATE_DIRS)
    raise FileNotFoundError(
        'Could not find whatsapp_train.csv, whatsapp_valid.csv, and whatsapp_test.csv. '\
        'Upload the prepared CSVs as a Kaggle dataset named whatsapp-fake-news.\nSearched:\n' + searched
    )

def validate_frame(df, split_name):
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f'{split_name} is missing required columns: {missing}')
    df = df[REQUIRED_COLUMNS].copy()
    df['text'] = df['text'].fillna('').astype(str)
    df['label'] = df['label'].astype(int)
    bad_labels = sorted(set(df['label'].unique()) - {0, 1})
    if bad_labels:
        raise ValueError(f'{split_name} has labels outside {{0, 1}}: {bad_labels}')
    return df

DATA_DIR = resolve_dataset_dir()
print(f'Using dataset directory: {DATA_DIR}')

train_df = validate_frame(pd.read_csv(DATA_DIR / 'whatsapp_train.csv'), 'train')
valid_df = validate_frame(pd.read_csv(DATA_DIR / 'whatsapp_valid.csv'), 'valid')
test_df = validate_frame(pd.read_csv(DATA_DIR / 'whatsapp_test.csv'), 'test')

for name, df in [('train', train_df), ('valid', valid_df), ('test', test_df)]:
    print(f'\n{name}: {len(df):,} rows')
    print('Label distribution:', df['label'].value_counts().sort_index().to_dict())
    print('Language distribution:', df['language'].value_counts().to_dict())
    print('Source distribution:', df['source_type'].value_counts().to_dict())
    print('Average text length:', round(df['text'].str.len().mean(), 2))

In [ ]:
FORWARDED_RE = re.compile(r'^\s*Forwarded\s+many\s+times\s*', flags=re.IGNORECASE)
URL_RE = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)
HANDLE_RE = re.compile(r'(?<!\w)@[A-Za-z0-9_]{2,}')
WHITESPACE_RE = re.compile(r'\s+')
# Emoji sequences like ⚠️ can leave variation selectors after the main emoji is removed.
EMOJI_MODIFIER_RE = re.compile(r'[\uFE0E\uFE0F\u200D]')
GURMUKHI_RE = re.compile(r'[\u0A00-\u0A7F]')
DEVANAGARI_RE = re.compile(r'[\u0900-\u097F]')
EMOJI_RE = re.compile(
    '['
    '\U0001F1E6-\U0001F1FF'
    '\U0001F300-\U0001F5FF'
    '\U0001F600-\U0001F64F'
    '\U0001F680-\U0001F6FF'
    '\U0001F700-\U0001F77F'
    '\U0001F780-\U0001F7FF'
    '\U0001F800-\U0001F8FF'
    '\U0001F900-\U0001F9FF'
    '\U0001FA70-\U0001FAFF'
    '\u2600-\u26FF'
    '\u2700-\u27BF'
    ']+' ,
    flags=re.UNICODE,
)

SHARE_KEYWORDS = [
    'जरूर शेयर', 'शेयर करें', 'आगे भेज', 'फॉरवर्ड', 'सबको भेज', 'सभी को भेज',
    'ਜ਼ਰੂਰ ਸ਼ੇਅਰ', 'ਸ਼ੇਅਰ ਕਰੋ', 'ਅੱਗੇ ਭੇਜ', 'ਸਭ ਨੂੰ ਭੇਜ',
    'share', 'forward', 'send everyone', 'must share'
]
POSITIVE_WORDS = {
    'सही', 'सच', 'अच्छा', 'लाभ', 'मदद', 'जीत', 'मुफ्त', 'फ्री', 'खुश', 'बधाई',
    'ਸੱਚ', 'ਚੰਗਾ', 'ਲਾਭ', 'ਮਦਦ', 'ਜਿੱਤ', 'ਮੁਫਤ', 'ਵਧਾਈ',
    'true', 'good', 'benefit', 'free', 'win', 'help'
}
NEGATIVE_WORDS = {
    'खतरा', 'झूठ', 'फेक', 'धोखा', 'बीमारी', 'मौत', 'घोटाला', 'डर', 'ब्लॉक', 'चेतावनी',
    'ਖਤਰਾ', 'ਝੂਠ', 'ਫੇਕ', 'ਧੋਖਾ', 'ਬਿਮਾਰੀ', 'ਮੌਤ', 'ਘੋਟਾਲਾ', 'ਡਰ', 'ਚੇਤਾਵਨੀ',
    'fake', 'fraud', 'danger', 'scam', 'death', 'warning', 'blocked'
}
METADATA_FEATURES = [
    'emoji_count',
    'exclamation_count',
    'text_length',
    'forwarded_flag',
    'share_words',
    'caps_ratio',
    'sentiment_score',
]

def count_emojis(text):
    return len(EMOJI_RE.findall(str(text)))

def remove_emojis(text):
    text = EMOJI_RE.sub(' ', str(text))
    return EMOJI_MODIFIER_RE.sub(' ', text)

def detect_language(text):
    text = str(text)
    return 'pa' if GURMUKHI_RE.search(text) else 'hi'

def caps_ratio(text):
    letters = [ch for ch in str(text) if ch.isalpha()]
    if not letters:
        return 0.0
    uppercase = sum(1 for ch in letters if ch.isupper())
    return uppercase / len(letters)

def count_share_keywords(text):
    lower = str(text).lower()
    return sum(lower.count(keyword.lower()) for keyword in SHARE_KEYWORDS)

def simple_sentiment_score(text):
    tokens = re.findall(r'[\w\u0900-\u097F\u0A00-\u0A7F]+', str(text).lower())
    if not tokens:
        return 0.0
    pos = sum(1 for token in tokens if token in POSITIVE_WORDS)
    neg = sum(1 for token in tokens if token in NEGATIVE_WORDS)
    return (pos - neg) / math.sqrt(len(tokens))

def clean_whatsapp_text(text):
    text = str(text)
    text = FORWARDED_RE.sub(' ', text)
    text = URL_RE.sub(' ', text)
    text = HANDLE_RE.sub(' ', text)
    text = remove_emojis(text)
    text = WHITESPACE_RE.sub(' ', text).strip()
    return text

def transformer_prefix(text):
    return '[PA]' if detect_language(text) == 'pa' else '[HI]'

def preprocess_dataframe(df):
    out = df.copy()
    original = out['text'].fillna('').astype(str)
    out['clean_text'] = original.map(clean_whatsapp_text)
    out['detected_language'] = original.map(detect_language)
    out['transformer_text'] = [f'{transformer_prefix(text)} {clean}' for text, clean in zip(original, out['clean_text'])]
    out['emoji_count'] = original.map(count_emojis)
    out['exclamation_count'] = original.map(lambda x: str(x).count('!') + str(x).count('！'))
    out['text_length'] = original.map(lambda x: len(str(x)))
    out['forwarded_flag'] = original.map(lambda x: int('forwarded' in str(x).lower()))
    out['share_words'] = original.map(count_share_keywords)
    out['caps_ratio'] = original.map(caps_ratio)
    out['sentiment_score'] = original.map(simple_sentiment_score)
    return out

train_df = preprocess_dataframe(train_df)
valid_df = preprocess_dataframe(valid_df)
test_df = preprocess_dataframe(test_df)

print(train_df[['text', 'clean_text', 'detected_language', 'transformer_text'] + METADATA_FEATURES].head(3))

In [ ]:
model_comparison = {}
valid_probabilities = {}
test_probabilities = {}
test_prediction_frame = test_df[['text', 'label', 'language', 'source_type', 'detected_language']].copy()

FAKE_LABEL = 1
REAL_LABEL = 0

def fake_probability_from_estimator(estimator, X):
    probabilities = estimator.predict_proba(X)
    classes = list(estimator.classes_)
    fake_index = classes.index(FAKE_LABEL)
    return probabilities[:, fake_index]

def metrics_from_probabilities(y_true, fake_probs, threshold=0.5):
    y_pred = (np.asarray(fake_probs) >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(precision),
        'recall_macro': float(recall),
        'f1_macro': float(f1),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
        'threshold': float(threshold),
    }

def register_model_result(model_name, y_true, fake_probs, threshold=0.5):
    metrics = metrics_from_probabilities(y_true, fake_probs, threshold=threshold)
    model_comparison[model_name] = metrics
    print(f'\n=== {model_name} ===')
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
    show_error_analysis(model_name, y_true, fake_probs, threshold=threshold)
    return metrics

def show_error_analysis(model_name, y_true, fake_probs, threshold=0.5, sample_size=5):
    y_true = np.asarray(y_true)
    fake_probs = np.asarray(fake_probs)
    y_pred = (fake_probs >= threshold).astype(int)
    error_df = test_df[['text', 'label', 'language', 'source_type']].copy()
    error_df['fake_probability'] = fake_probs
    false_positives = error_df[(y_true == REAL_LABEL) & (y_pred == FAKE_LABEL)].head(sample_size)
    false_negatives = error_df[(y_true == FAKE_LABEL) & (y_pred == REAL_LABEL)].head(sample_size)
    print('\nFalse positives: real messages flagged fake')
    display(false_positives.assign(text=false_positives['text'].str.slice(0, 220)))
    print('\nFalse negatives: fake messages missed')
    display(false_negatives.assign(text=false_negatives['text'].str.slice(0, 220)))

def save_json(path, obj):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

In [ ]:
if RUN_CLASSICAL:
    print('Training classical models...')
    tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
    X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['clean_text'])
    X_valid_tfidf = tfidf_vectorizer.transform(valid_df['clean_text'])
    X_test_tfidf = tfidf_vectorizer.transform(test_df['clean_text'])

    y_train = train_df['label'].to_numpy()
    y_valid = valid_df['label'].to_numpy()
    y_test = test_df['label'].to_numpy()

    logistic_regression = LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear', random_state=SEED)
    logistic_regression.fit(X_train_tfidf, y_train)
    lr_valid_probs = fake_probability_from_estimator(logistic_regression, X_valid_tfidf)
    lr_test_probs = fake_probability_from_estimator(logistic_regression, X_test_tfidf)
    valid_probabilities['logistic_regression'] = lr_valid_probs
    test_probabilities['logistic_regression'] = lr_test_probs
    test_prediction_frame['logistic_regression_fake_prob'] = lr_test_probs
    register_model_result('logistic_regression', y_test, lr_test_probs)

    naive_bayes = MultinomialNB()
    naive_bayes.fit(X_train_tfidf, y_train)
    nb_valid_probs = fake_probability_from_estimator(naive_bayes, X_valid_tfidf)
    nb_test_probs = fake_probability_from_estimator(naive_bayes, X_test_tfidf)
    valid_probabilities['naive_bayes'] = nb_valid_probs
    test_probabilities['naive_bayes'] = nb_test_probs
    test_prediction_frame['naive_bayes_fake_prob'] = nb_test_probs
    register_model_result('naive_bayes', y_test, nb_test_probs)

    metadata_scaler = StandardScaler()
    train_meta = metadata_scaler.fit_transform(train_df[METADATA_FEATURES].astype(float))
    valid_meta = metadata_scaler.transform(valid_df[METADATA_FEATURES].astype(float))
    test_meta = metadata_scaler.transform(test_df[METADATA_FEATURES].astype(float))

    X_train_xgb = sparse.hstack([X_train_tfidf, sparse.csr_matrix(train_meta)], format='csr')
    X_valid_xgb = sparse.hstack([X_valid_tfidf, sparse.csr_matrix(valid_meta)], format='csr')
    X_test_xgb = sparse.hstack([X_test_tfidf, sparse.csr_matrix(test_meta)], format='csr')

    xgboost_model = XGBClassifier(
        n_estimators=350,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1,
    )
    xgboost_model.fit(X_train_xgb, y_train, eval_set=[(X_valid_xgb, y_valid)], verbose=False)
    xgb_valid_probs = fake_probability_from_estimator(xgboost_model, X_valid_xgb)
    xgb_test_probs = fake_probability_from_estimator(xgboost_model, X_test_xgb)
    valid_probabilities['xgboost'] = xgb_valid_probs
    test_probabilities['xgboost'] = xgb_test_probs
    test_prediction_frame['xgboost_fake_prob'] = xgb_test_probs
    register_model_result('xgboost', y_test, xgb_test_probs)

    joblib.dump(tfidf_vectorizer, CLASSICAL_DIR / 'tfidf_vectorizer.joblib')
    joblib.dump(logistic_regression, CLASSICAL_DIR / 'logistic_regression.joblib')
    joblib.dump(naive_bayes, CLASSICAL_DIR / 'naive_bayes.joblib')
    joblib.dump(metadata_scaler, CLASSICAL_DIR / 'metadata_scaler.joblib')
    joblib.dump(xgboost_model, CLASSICAL_DIR / 'xgboost.joblib')
    save_json(CLASSICAL_DIR / 'feature_config.json', {'metadata_features': METADATA_FEATURES})
else:
    print('RUN_CLASSICAL=False, skipping classical models.')

In [ ]:
def make_training_arguments(output_dir, alias):
    kwargs = {
        'output_dir': str(output_dir),
        'learning_rate': LEARNING_RATE,
        'per_device_train_batch_size': BATCH_SIZE,
        'per_device_eval_batch_size': BATCH_SIZE,
        'num_train_epochs': EPOCHS,
        'weight_decay': 0.01,
        'logging_steps': 100,
        'save_total_limit': 1,
        'load_best_model_at_end': True,
        'metric_for_best_model': 'f1_macro',
        'greater_is_better': True,
        'report_to': 'none',
        'seed': SEED,
        'fp16': bool(torch.cuda.is_available()),
    }
    signature = inspect.signature(TrainingArguments.__init__)
    if 'eval_strategy' in signature.parameters:
        kwargs['eval_strategy'] = 'epoch'
    else:
        kwargs['evaluation_strategy'] = 'epoch'
    kwargs['save_strategy'] = 'epoch'
    return TrainingArguments(**kwargs)

def make_tokenized_dataset(df, tokenizer):
    source = df[['transformer_text', 'label']].rename(columns={'label': 'labels'}).copy()
    dataset = Dataset.from_pandas(source, preserve_index=False)

    def tokenize(batch):
        return tokenizer(
            batch['transformer_text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH,
        )

    tokenized = dataset.map(tokenize, batched=True, remove_columns=['transformer_text'])
    tokenized.set_format(type='torch')
    return tokenized

def compute_transformer_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='macro', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'precision_macro': float(precision),
        'recall_macro': float(recall),
        'f1_macro': float(f1),
    }

def softmax_numpy(logits):
    logits = np.asarray(logits)
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / np.sum(exp, axis=1, keepdims=True)

def train_one_transformer(alias, candidate_model_ids):
    last_error = None
    for model_id in candidate_model_ids:
        model_output_dir = TRANSFORMER_DIR / alias
        run_output_dir = SAVE_DIR / 'trainer_runs' / alias
        try:
            print(f'\nTraining {alias} from {model_id}')
            tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
            model = AutoModelForSequenceClassification.from_pretrained(
                model_id,
                num_labels=2,
                id2label={0: 'REAL', 1: 'FAKE'},
                label2id={'REAL': 0, 'FAKE': 1},
                ignore_mismatched_sizes=True,
            )

            tokenized_train = make_tokenized_dataset(train_df, tokenizer)
            tokenized_valid = make_tokenized_dataset(valid_df, tokenizer)
            tokenized_test = make_tokenized_dataset(test_df, tokenizer)

            args = make_training_arguments(run_output_dir, alias)
            trainer_kwargs = {
                'model': model,
                'args': args,
                'train_dataset': tokenized_train,
                'eval_dataset': tokenized_valid,
                'compute_metrics': compute_transformer_metrics,
            }
            trainer_signature = inspect.signature(Trainer.__init__)
            if 'processing_class' in trainer_signature.parameters:
                trainer_kwargs['processing_class'] = tokenizer
            else:
                trainer_kwargs['tokenizer'] = tokenizer
            trainer = Trainer(**trainer_kwargs)
            trainer.train()
            trainer.save_model(model_output_dir)
            tokenizer.save_pretrained(model_output_dir)

            valid_output = trainer.predict(tokenized_valid)
            test_output = trainer.predict(tokenized_test)
            valid_probs = softmax_numpy(valid_output.predictions)[:, 1]
            test_probs = softmax_numpy(test_output.predictions)[:, 1]

            valid_probabilities[alias] = valid_probs
            test_probabilities[alias] = test_probs
            test_prediction_frame[f'{alias}_fake_prob'] = test_probs
            metrics = register_model_result(alias, test_df['label'].to_numpy(), test_probs)
            metrics['source_model_id'] = model_id
            model_comparison[alias] = metrics
            save_json(model_output_dir / 'training_metadata.json', {
                'alias': alias,
                'source_model_id': model_id,
                'max_length': MAX_LENGTH,
                'epochs': EPOCHS,
                'learning_rate': LEARNING_RATE,
                'batch_size': BATCH_SIZE,
            })

            del trainer, model, tokenizer, tokenized_train, tokenized_valid, tokenized_test
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return True
        except Exception as exc:
            last_error = repr(exc)
            print(f'Failed {alias} candidate {model_id}: {last_error}')
            model_comparison[f'{alias}__{model_id}__error'] = {'error': last_error}
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    print(f'All candidates failed for {alias}. Last error: {last_error}')
    return False

In [ ]:
TRANSFORMER_SPECS = {
    'indic_bert': ['ai4bharat/indic-bert'],
    'punjabi_bert': ['neuralspace-reverie/indic-transformers-pa-bert', 'l3cube-pune/punjabi-bert'],
    'muril': ['google/muril-base-cased'],
}

if RUN_TRANSFORMERS:
    for alias, candidates in TRANSFORMER_SPECS.items():
        train_one_transformer(alias, candidates)
else:
    print('RUN_TRANSFORMERS=False, skipping transformer fine-tuning.')

In [ ]:
if TRAIN_STACKING:
    print('\nTraining stacking ensemble...')
    stack_feature_names = []
    valid_parts = []
    test_parts = []

    for name in sorted(valid_probabilities.keys()):
        if name in test_probabilities:
            stack_feature_names.append(f'{name}_fake_prob')
            valid_parts.append(np.asarray(valid_probabilities[name]).reshape(-1, 1))
            test_parts.append(np.asarray(test_probabilities[name]).reshape(-1, 1))

    if 'metadata_scaler' in globals():
        valid_meta_stack = metadata_scaler.transform(valid_df[METADATA_FEATURES].astype(float))
        test_meta_stack = metadata_scaler.transform(test_df[METADATA_FEATURES].astype(float))
    else:
        metadata_scaler = StandardScaler()
        valid_meta_stack = metadata_scaler.fit_transform(valid_df[METADATA_FEATURES].astype(float))
        test_meta_stack = metadata_scaler.transform(test_df[METADATA_FEATURES].astype(float))
        joblib.dump(metadata_scaler, ENSEMBLE_DIR / 'metadata_scaler_for_stack.joblib')

    stack_feature_names.extend(METADATA_FEATURES)
    valid_parts.append(valid_meta_stack)
    test_parts.append(test_meta_stack)

    if len(valid_parts) >= 2:
        X_stack_valid = np.column_stack(valid_parts)
        X_stack_test = np.column_stack(test_parts)
        stacker = XGBClassifier(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='binary:logistic',
            eval_metric='logloss',
            tree_method='hist',
            random_state=SEED,
            n_jobs=-1,
        )
        stacker.fit(X_stack_valid, valid_df['label'].to_numpy(), verbose=False)
        ensemble_test_probs = fake_probability_from_estimator(stacker, X_stack_test)
        test_probabilities['stacking_ensemble'] = ensemble_test_probs
        test_prediction_frame['stacking_ensemble_fake_prob'] = ensemble_test_probs
        register_model_result('stacking_ensemble', test_df['label'].to_numpy(), ensemble_test_probs)
        joblib.dump(stacker, ENSEMBLE_DIR / 'stacking_xgboost.joblib')
        save_json(ENSEMBLE_DIR / 'stacking_feature_config.json', {'feature_names': stack_feature_names})
    else:
        print('Not enough base predictions for stacking ensemble. Skipping.')
        model_comparison['stacking_ensemble'] = {'error': 'Not enough base predictions for stacking ensemble.'}
else:
    print('TRAIN_STACKING=False, skipping stacking ensemble.')

In [ ]:
# Final artifact files
training_config = {
    'dataset_dir': str(DATA_DIR),
    'seed': SEED,
    'max_length': MAX_LENGTH,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'device': DEVICE,
    'classical_models': ['logistic_regression', 'naive_bayes', 'xgboost'],
    'transformer_specs': TRANSFORMER_SPECS,
    'metadata_features': METADATA_FEATURES,
    'label_map': {'0': 'REAL', '1': 'FAKE'},
    'created_at_unix': int(time.time()),
}

# Add hard predictions for convenience.
for column in [col for col in test_prediction_frame.columns if col.endswith('_fake_prob')]:
    pred_column = column.replace('_fake_prob', '_prediction')
    test_prediction_frame[pred_column] = np.where(test_prediction_frame[column] >= 0.5, 'FAKE', 'REAL')

test_prediction_frame.to_csv(SAVE_DIR / 'test_predictions.csv', index=False)
save_json(SAVE_DIR / 'model_comparison.json', model_comparison)
save_json(SAVE_DIR / 'training_config.json', training_config)

zip_base = shutil.make_archive(str(SAVE_DIR), 'zip', root_dir=SAVE_DIR)
print(f'\nSaved model bundle: {zip_base}')
print('\nArtifact directory contents:')
for path in sorted(SAVE_DIR.rglob('*')):
    print(path.relative_to(SAVE_DIR))